# Experimento: Gradiente Evolutivo SW → HW → Grass
## Notebook 03 — Ejecución con réplicas y registro de t_gen

**Hipótesis:** Si el gradiente evolutivo SW → HW → Grass se refleja en t_gen:
> t_gen(SW) > t_gen(HW) > t_gen(GR) para DP equivalente

**Diseño:** 3 grupos × 3 niveles de DP × 30 réplicas = 270 corridas

**Prerequisito:** Copiar los 9 YAMLs de `experiment_yamls/` a `resources/`

In [ ]:
# ============================================================
# CELDA 1 — CONFIGURACIÓN
# Ajusta BASE_PATH a tu ruta real
# ============================================================

import os
import subprocess
import shutil
import time
import glob
import csv
from datetime import datetime

# ── AJUSTA ESTA RUTA ──────────────────────────────────────
BASE_PATH = "/media/aldo/Aldo/sims/Lignin_DB_Chemically_Real-5"
# ──────────────────────────────────────────────────────────

N_REPLICAS = 30  # réplicas por configuración

JAR_PATH       = os.path.join(BASE_PATH, "lgs_gen.jar")
RESOURCES_DIR  = os.path.join(BASE_PATH, "resources")
OUTPUT_DIR     = os.path.join(BASE_PATH, "output")
RESULTS_CSV    = os.path.join(BASE_PATH, "evolutionary_experiment_results.csv")

# Verificar que el jar existe
assert os.path.exists(JAR_PATH), f"❌ No encontré lgs_gen.jar en {JAR_PATH}"

# Los 9 YAMLs del experimento deben estar en resources/
EXPERIMENT_CONFIGS = sorted(glob.glob(os.path.join(RESOURCES_DIR, "SW_*_config.yaml")) +
                            glob.glob(os.path.join(RESOURCES_DIR, "HW_*_config.yaml")) +
                            glob.glob(os.path.join(RESOURCES_DIR, "GR_*_config.yaml")))

print(f"✅ lgs_gen.jar encontrado")
print(f"📋 Configuraciones del experimento ({len(EXPERIMENT_CONFIGS)} encontradas):")
for f in EXPERIMENT_CONFIGS:
    print(f"   • {os.path.basename(f)}")

if len(EXPERIMENT_CONFIGS) != 9:
    print(f"\n⚠️  Se esperaban 9 YAMLs. Verifica que copiaste los archivos a resources/")
else:
    print(f"\n✅ Todo listo. El experimento correrá {len(EXPERIMENT_CONFIGS)} × {N_REPLICAS} = {len(EXPERIMENT_CONFIGS)*N_REPLICAS} corridas")

In [ ]:
# ============================================================
# CELDA 2 — FUNCIÓN DE CORRIDA ÚNICA
# Igual al notebook original pero devuelve t_gen explícitamente
# ============================================================

def run_lgs_once(config_file, replica_num, base_path, jar_path, resources_dir, output_dir):
    """
    Corre LGS UNA vez con un config dado.
    Devuelve t_gen en segundos y si fue exitoso.
    Los outputs se borran después de cada réplica para no llenar el disco.
    """
    config_name = os.path.basename(config_file).replace('_config.yaml', '')
    project_config = os.path.join(resources_dir, "project_config.yaml")

    # Backup del project_config si existe
    backup = None
    if os.path.exists(project_config):
        backup = project_config + ".backup"
        shutil.move(project_config, backup)

    t_gen = None
    success = False
    error_msg = ""

    try:
        # Copiar el YAML del experimento como project_config.yaml
        shutil.copy2(config_file, project_config)

        # ── MEDICIÓN DE t_gen ──────────────────────────────
        t_start = time.time()
        result = subprocess.run(
            ["java", "-jar", jar_path],
            capture_output=True,
            text=True,
            cwd=base_path
        )
        t_end = time.time()
        # ───────────────────────────────────────────────────

        t_gen = t_end - t_start
        success = (result.returncode == 0)
        if not success:
            error_msg = result.stderr[:200]

    except Exception as e:
        error_msg = str(e)

    finally:
        # Limpiar project_config
        if os.path.exists(project_config):
            os.remove(project_config)
        if backup and os.path.exists(backup):
            shutil.move(backup, project_config)

        # Borrar outputs de esta réplica para no llenar el disco
        # (solo guardamos el t_gen, no las estructuras)
        for folder in ['json', 'mol', 'struct', 'matrices']:
            folder_path = os.path.join(output_dir, folder)
            if os.path.exists(folder_path):
                shutil.rmtree(folder_path)

    return {
        'config':    config_name,
        'replica':   replica_num,
        't_gen':     t_gen,
        'success':   success,
        'error':     error_msg,
        'timestamp': datetime.now().isoformat()
    }

print("✅ Función run_lgs_once definida")

In [ ]:
# ============================================================
# CELDA 3 — PRUEBA CON 1 RÉPLICA ANTES DE CORRER TODO
# Verifica que todo funciona antes de lanzar las 270 corridas
# ============================================================

print("🧪 Corriendo prueba con 1 réplica de SW_DP5...")

test_config = [f for f in EXPERIMENT_CONFIGS if 'SW_DP5' in f]
if not test_config:
    print("❌ No encontré SW_DP5_config.yaml en resources/")
else:
    test_result = run_lgs_once(
        config_file   = test_config[0],
        replica_num   = 0,
        base_path     = BASE_PATH,
        jar_path      = JAR_PATH,
        resources_dir = RESOURCES_DIR,
        output_dir    = OUTPUT_DIR
    )
    if test_result['success']:
        print(f"✅ Prueba exitosa")
        print(f"   Config:  {test_result['config']}")
        print(f"   t_gen:   {test_result['t_gen']:.2f}s")
        print(f"\n🚀 Listo para correr el experimento completo (Celda 4)")
    else:
        print(f"❌ Prueba fallida: {test_result['error']}")
        print("   Revisa BASE_PATH y que lgs_gen.jar funcione correctamente")

In [ ]:
# ============================================================
# CELDA 4 — EXPERIMENTO COMPLETO
# 9 configuraciones × 30 réplicas = 270 corridas
# Guarda resultados en CSV después de cada réplica
# (si se interrumpe, los datos ya guardados no se pierden)
# ============================================================

# Encabezado del CSV — metadatos del experimento incluidos
CSV_FIELDS = ['config', 'group', 'dp', 'fBO4', 'sg_ratio',
              'replica', 't_gen', 'success', 'error', 'timestamp']

# Tabla de metadatos por configuración (para el CSV)
CONFIG_META = {
    'SW_DP5':  {'group': 'SW', 'dp': 5,  'fBO4': 0.54, 'sg_ratio': 0.0},
    'SW_DP8':  {'group': 'SW', 'dp': 8,  'fBO4': 0.54, 'sg_ratio': 0.0},
    'SW_DP10': {'group': 'SW', 'dp': 10, 'fBO4': 0.54, 'sg_ratio': 0.0},
    'HW_DP5':  {'group': 'HW', 'dp': 5,  'fBO4': 0.69, 'sg_ratio': 1.5},
    'HW_DP8':  {'group': 'HW', 'dp': 8,  'fBO4': 0.69, 'sg_ratio': 1.5},
    'HW_DP10': {'group': 'HW', 'dp': 10, 'fBO4': 0.69, 'sg_ratio': 1.5},
    'GR_DP5':  {'group': 'GR', 'dp': 5,  'fBO4': 0.77, 'sg_ratio': 0.8},
    'GR_DP8':  {'group': 'GR', 'dp': 8,  'fBO4': 0.77, 'sg_ratio': 0.8},
    'GR_DP10': {'group': 'GR', 'dp': 10, 'fBO4': 0.77, 'sg_ratio': 0.8},
}

# Abrir CSV (modo append para no perder datos si se interrumpe)
csv_exists = os.path.exists(RESULTS_CSV)
csv_file = open(RESULTS_CSV, 'a', newline='')
writer = csv.DictWriter(csv_file, fieldnames=CSV_FIELDS)
if not csv_exists:
    writer.writeheader()

# ── BUCLE PRINCIPAL ───────────────────────────────────────
total = len(EXPERIMENT_CONFIGS) * N_REPLICAS
done  = 0
t_experiment_start = time.time()

print(f"🚀 INICIANDO EXPERIMENTO EVOLUTIVO")
print(f"   Configuraciones: {len(EXPERIMENT_CONFIGS)}")
print(f"   Réplicas por config: {N_REPLICAS}")
print(f"   Total de corridas: {total}")
print(f"   Resultados → {RESULTS_CSV}")
print("-" * 60)

for config_file in EXPERIMENT_CONFIGS:
    config_name = os.path.basename(config_file).replace('_config.yaml', '')
    meta = CONFIG_META.get(config_name, {})

    print(f"\n📦 {config_name}  (fBO4={meta.get('fBO4','?')}, S/G={meta.get('sg_ratio','?')})")
    t_gen_list = []

    for rep in range(1, N_REPLICAS + 1):
        result = run_lgs_once(
            config_file   = config_file,
            replica_num   = rep,
            base_path     = BASE_PATH,
            jar_path      = JAR_PATH,
            resources_dir = RESOURCES_DIR,
            output_dir    = OUTPUT_DIR
        )

        # Agregar metadatos al resultado
        row = {
            'config':    result['config'],
            'group':     meta.get('group', ''),
            'dp':        meta.get('dp', ''),
            'fBO4':      meta.get('fBO4', ''),
            'sg_ratio':  meta.get('sg_ratio', ''),
            'replica':   result['replica'],
            't_gen':     round(result['t_gen'], 4) if result['t_gen'] else '',
            'success':   result['success'],
            'error':     result['error'],
            'timestamp': result['timestamp']
        }
        writer.writerow(row)
        csv_file.flush()  # guardar inmediatamente

        if result['success'] and result['t_gen']:
            t_gen_list.append(result['t_gen'])

        done += 1
        pct = done / total * 100
        symbol = '✅' if result['success'] else '❌'
        t_str  = f"{result['t_gen']:.1f}s" if result['t_gen'] else 'N/A'
        print(f"   {symbol} Rep {rep:02d}/{N_REPLICAS}  t_gen={t_str}  [{pct:.0f}% global]",
              end='\r')

    # Resumen de la configuración
    if t_gen_list:
        mean_t = sum(t_gen_list) / len(t_gen_list)
        sd_t   = (sum((x - mean_t)**2 for x in t_gen_list) / len(t_gen_list))**0.5
        print(f"   ✅ {len(t_gen_list)}/{N_REPLICAS} exitosas  "
              f"t_gen = {mean_t:.2f} ± {sd_t:.2f} s")

csv_file.close()

t_total = time.time() - t_experiment_start
print("\n" + "=" * 60)
print(f"✅ EXPERIMENTO COMPLETO")
print(f"   Tiempo total: {t_total/60:.1f} minutos")
print(f"   Resultados guardados en: {RESULTS_CSV}")

In [ ]:
# ============================================================
# CELDA 5 — RESUMEN RÁPIDO DE RESULTADOS
# Corre esta celda después del experimento para ver si
# el gradiente t_gen(SW) > t_gen(HW) > t_gen(GR) se cumple
# ============================================================

import csv
from collections import defaultdict

data = defaultdict(list)

with open(RESULTS_CSV, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row['success'] == 'True' and row['t_gen']:
            key = (row['group'], int(row['dp']))
            data[key].append(float(row['t_gen']))

print(f"{'Grupo':<6} {'DP':<5} {'n':>4} {'media (s)':>10} {'SD':>8} {'min':>8} {'max':>8}")
print("-" * 52)

for group in ['SW', 'HW', 'GR']:
    for dp in [5, 8, 10]:
        vals = data.get((group, dp), [])
        if vals:
            mean = sum(vals) / len(vals)
            sd   = (sum((x-mean)**2 for x in vals)/len(vals))**0.5
            print(f"{group:<6} {dp:<5} {len(vals):>4} "
                  f"{mean:>10.2f} {sd:>8.2f} "
                  f"{min(vals):>8.2f} {max(vals):>8.2f}")

print()
print("Gradiente esperado (hipótesis): t_gen(SW) > t_gen(HW) > t_gen(GR)")

# Verificar el ordenamiento por DP
for dp in [5, 8, 10]:
    vals = {g: data.get((g, dp), []) for g in ['SW', 'HW', 'GR']}
    if all(vals.values()):
        means = {g: sum(v)/len(v) for g, v in vals.items()}
        ordered = means['SW'] > means['HW'] > means['GR']
        symbol = '✅' if ordered else '❌'
        print(f"  DP={dp}: SW={means['SW']:.1f}s  HW={means['HW']:.1f}s  "
              f"GR={means['GR']:.1f}s  →  {symbol} {'CUMPLE' if ordered else 'NO CUMPLE'}")